Predicting House Prices by Jay and Derick:

How to run:
1. open in Google Colab
2. download train and test files from kaggle into your google drive, make sure that they're labeled as "test.csv" and "train.csv" so that when your drive is linked, the right files are opened and rest of code works smoothly
3. run code from top to bottom

Approach:
1. Dropped columns with majority of their values missing as filling in too many of the values with a median or something else along those lines would bias the data too much
2. Filled in rest of columns with missing values either with a 'None' (for columns like basement quality when the house is missing that value because they don't have a basement), the median value (for columns like garage year built that can't be zero or none because it'll skew the data way too much for a house that just doesn't have a garage) , or zero (for columns like Masonry veneer area where if a house doesn't have one, it makes sense that the area would be 0).
3. Changed columns that had sequential orders like basement quality into numbers that were ranked using ordinal encoder. Treated none as being worse than poor becaues assumed that having a poor basement is still better than having no basement.
4. Changed rest of categorical columns into numbers using onehot encoder.
5. Made sure test and train data were transformed in same manner and then used regression to come up with a "submissions.csv" that contains all predicted house prices.



In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [3]:
#import data
test = pd.read_csv('/content/drive/MyDrive/test.csv')

In [4]:
train = pd.read_csv('/content/drive/MyDrive/train.csv')

In [5]:
train.head(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500


In [6]:
#find columns with missing values
train.columns[train.isna().sum() > 0]

Index(['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish',
       'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'],
      dtype='object')

In [7]:
#find % to standardize and see which columns can be dropped
train[train.columns[train.isna().sum() > 0]].isna().sum()/len(train)

,0
LotFrontage,0.175000
Alley,0.937500
MasVnrType,0.597500
MasVnrArea,0.005000
BsmtQual,0.026667
BsmtCond,0.026667
BsmtExposure,0.027500
BsmtFinType1,0.026667
BsmtFinType2,0.027500
FireplaceQu,0.470000


In [8]:
#dropping columns with nearly all their data missing
train = train.drop(columns = ['PoolQC','Alley','MiscFeature','Fence'])

In [9]:
#columns successfully dropped
train.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'WoodD

In [10]:
#creating df with all columns with missing values
missing = train[train.columns[train.isna().sum() > 0]]

In [11]:
#for columns like MasVnrType missing value should be filled with none instead of most frequent bc house doesn't have it
missing

,LotFrontage,MasVnrType,MasVnrArea,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageQual,GarageCond
0,65.0,BrkFace,196.0,Gd,TA,No,GLQ,Unf,NaN,Attchd,2003.0,RFn,TA,TA
1,80.0,NaN,0.0,Gd,TA,Gd,ALQ,Unf,TA,Attchd,1976.0,RFn,TA,TA
2,68.0,BrkFace,162.0,Gd,TA,Mn,GLQ,Unf,TA,Attchd,2001.0,RFn,TA,TA
3,60.0,NaN,0.0,TA,Gd,No,ALQ,Unf,Gd,Detchd,1998.0,Unf,TA,TA
4,84.0,BrkFace,350.0,Gd,TA,Av,GLQ,Unf,TA,Attchd,2000.0,RFn,TA,TA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,51.0,NaN,0.0,Gd,TA,No,Unf,Unf,NaN,Attchd,2005.0,Fin,TA,TA
1196,58.0,NaN,0.0,Gd,TA,Av,Unf,Unf,Gd,BuiltIn,2006.0,Fin,TA,TA
1197,65.0,NaN,0.0,TA,TA,No,Unf,Unf,Gd,Detchd,1916.0,Unf,TA,TA
1198,70.0,NaN,0.0,Gd,TA,No,Unf,Unf,NaN,Attchd,2001.0,RFn,TA,TA


In [12]:
#picking which columns are going to have their missing vales replaced with median, 'None', or 0
median_cols = ['LotFrontage','GarageYrBlt']
none_cols = ['MasVnrType','BsmtQual', 'BsmtCond',	'BsmtExposure',	'BsmtFinType1', 'BsmtFinType2',
             'FireplaceQu','GarageType', 'GarageFinish',	'GarageQual',	'GarageCond']
zero_cols = ['MasVnrArea']

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

In [14]:
#creating the imputer
imputer = ColumnTransformer([
    ('median', SimpleImputer(strategy='median', add_indicator=True), median_cols),
    ('zero',   SimpleImputer(strategy='constant', fill_value=0), zero_cols),
    ('none',   SimpleImputer(strategy='constant', fill_value='None'), none_cols),
], remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

In [15]:
#setting variables, dropping saleprice bc that's the target variable and dropping Id bc doesn't acc add any value to the prediction
x = train.drop(columns=['SalePrice', 'Id'])
y = train['SalePrice']

In [16]:
#imputing and checking to ensure that there are no missing values
x_imp = imputer.fit_transform(x)
print(x_imp.isna().sum().sum())

0


In [17]:
#updated df
x_imp

,LotFrontage,GarageYrBlt,missingindicator_LotFrontage,missingindicator_GarageYrBlt,MasVnrArea,MasVnrType,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,65.0,2003.0,0.0,0.0,196.0,BrkFace,Gd,TA,No,GLQ,...,61,0,0,0,0,0,2,2008,WD,Normal
1,80.0,1976.0,0.0,0.0,0.0,None,Gd,TA,Gd,ALQ,...,0,0,0,0,0,0,5,2007,WD,Normal
2,68.0,2001.0,0.0,0.0,162.0,BrkFace,Gd,TA,Mn,GLQ,...,42,0,0,0,0,0,9,2008,WD,Normal
3,60.0,1998.0,0.0,0.0,0.0,None,TA,Gd,No,ALQ,...,35,272,0,0,0,0,2,2006,WD,Abnorml
4,84.0,2000.0,0.0,0.0,350.0,BrkFace,Gd,TA,Av,GLQ,...,84,0,0,0,0,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,51.0,2005.0,0.0,0.0,0.0,None,Gd,TA,No,Unf,...,24,0,0,0,0,0,7,2008,WD,Normal
1196,58.0,2006.0,0.0,0.0,0.0,None,Gd,TA,Av,Unf,...,17,0,0,0,0,0,11,2006,New,Partial
1197,65.0,1916.0,0.0,0.0,0.0,None,TA,TA,No,Unf,...,0,330,0,0,0,0,7,2006,ConLw,Normal
1198,70.0,2001.0,0.0,0.0,0.0,None,Gd,TA,No,Unf,...,0,0,0,0,0,0,6,2009,WD,Normal


In [18]:
#finding all columns that aren't an integer
cat_cols = x.select_dtypes(exclude='integer').columns.tolist()
cat_cols

['MSZoning',
 'LotFrontage',
 'Street',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'MasVnrArea',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 'KitchenQual',
 'Functional',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'SaleType',
 'SaleCondition']

In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

In [20]:
#finding all columns with any of the following 'Po','Fa', etc. in their column bc will ordinal encode those columns
vals = ['None','Po', 'Fa', 'TA', 'Gd', 'Ex']

oe_cols = x_imp.columns[x_imp.isin(vals).any()].tolist()
print(oe_cols)


['MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']


In [21]:
#defining the orders for each column and ordinal encoding them, including none in there bc we added that in to fill in missing values and will assume that
#not having smth is worse than having that smth but it's poor quality
from sklearn.preprocessing import OrdinalEncoder

qual = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

orders = {
    'BsmtQual': qual, 'BsmtCond': qual, 'FireplaceQu': qual,
    'GarageQual': qual, 'GarageCond': qual,
    'ExterQual': qual, 'ExterCond': qual, 'HeatingQC': qual, 'KitchenQual': qual,
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
}

ordinal_cols = [c for c in orders if c in x_imp.columns]

oe = OrdinalEncoder(categories=[orders[c] for c in ordinal_cols])
x_imp[ordinal_cols] = oe.fit_transform(x_imp[ordinal_cols])


In [22]:
#updated df
x_imp

,LotFrontage,GarageYrBlt,missingindicator_LotFrontage,missingindicator_GarageYrBlt,MasVnrArea,MasVnrType,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,65.0,2003.0,0.0,0.0,196.0,BrkFace,4.0,3.0,1.0,6.0,...,61,0,0,0,0,0,2,2008,WD,Normal
1,80.0,1976.0,0.0,0.0,0.0,None,4.0,3.0,4.0,5.0,...,0,0,0,0,0,0,5,2007,WD,Normal
2,68.0,2001.0,0.0,0.0,162.0,BrkFace,4.0,3.0,2.0,6.0,...,42,0,0,0,0,0,9,2008,WD,Normal
3,60.0,1998.0,0.0,0.0,0.0,None,3.0,4.0,1.0,5.0,...,35,272,0,0,0,0,2,2006,WD,Abnorml
4,84.0,2000.0,0.0,0.0,350.0,BrkFace,4.0,3.0,3.0,6.0,...,84,0,0,0,0,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,51.0,2005.0,0.0,0.0,0.0,None,4.0,3.0,1.0,1.0,...,24,0,0,0,0,0,7,2008,WD,Normal
1196,58.0,2006.0,0.0,0.0,0.0,None,4.0,3.0,3.0,1.0,...,17,0,0,0,0,0,11,2006,New,Partial
1197,65.0,1916.0,0.0,0.0,0.0,None,3.0,3.0,1.0,1.0,...,0,330,0,0,0,0,7,2006,ConLw,Normal
1198,70.0,2001.0,0.0,0.0,0.0,None,4.0,3.0,1.0,1.0,...,0,0,0,0,0,0,6,2009,WD,Normal


In [23]:
ohe_cols = x_imp.select_dtypes(exclude=['integer','float']).columns.tolist()

In [24]:
#columns that aren't ordinal so will have to onehotencode
ohe_cols

['MasVnrType',
 'GarageType',
 'MSZoning',
 'Street',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'Foundation',
 'Heating',
 'CentralAir',
 'Electrical',
 'Functional',
 'PavedDrive',
 'SaleType',
 'SaleCondition']

In [25]:
#using onehot to transform remaining categorical columns into 1s and 0s, dropping first bc redundant
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ohe = ColumnTransformer(
    [('onehot', OneHotEncoder(drop='first',handle_unknown='ignore', sparse_output=False), ohe_cols)],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

x_enc = ohe.fit_transform(x_imp)


In [26]:
#finalized df
x_enc.head()

,MasVnrType_BrkFace,MasVnrType_None,MasVnrType_Stone,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None,MSZoning_FV,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,548,0,61,0,0,0,0,0,2,2008
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,460,298,0,0,0,0,0,0,5,2007
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,608,0,42,0,0,0,0,0,9,2008
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,642,0,35,272,0,0,0,0,2,2006
4,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,836,192,84,0,0,0,0,0,12,2008


In [27]:
#everything is a number so done "fixing" data
x_enc.select_dtypes(exclude=['integer','float']).columns.tolist()

[]

In [46]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge


#make sure test data set has same columns as train, saving 'id' column bc need it in submission csv
test_ids = test['Id']
test_x = test[x.columns]

#transforming data, making sure test and train are transformed same way, but only fit train data not test to avoid data leakage
x_imp = imputer.fit_transform(x)
test_imp = imputer.transform(test_x)

#filling in any blanks with none bc means they just don't have it, same process we did with train data
test_imp[ordinal_cols] = test_imp[ordinal_cols].fillna('None')
test_imp[ordinal_cols] = oe.transform(test_imp[ordinal_cols])

# fill any test-only blanks in those columns with the training mode
for c in ohe_cols:
    test_imp[c] = test_imp[c].fillna(x_imp[c].mode()[0])

test_enc = ohe.transform(test_imp)
test_enc = test_enc[x_enc.columns]

#making pipeline and imputing with median, standard scaling and using ridge to reduce overfitting and help with correlated features
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    Ridge(alpha=10)
)

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 14, 15, 16, 17, 19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [41]:
#creating predictions
model.fit(x_enc, y)
preds = model.predict(test_enc).clip(min=0)   #.clip ensures that there are no predictions have a negative value

In [42]:
#creating submission df
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})

#match the Id with index to adhere to submission guidlines
submission['Id'] = submission.index

In [43]:
#creating submission csv
submission.to_csv('submission.csv', index=False)


In [44]:
submission.head()

,Id,SalePrice
0,0,73196.568324
1,1,203229.316196
2,2,132676.229015
3,3,191806.991787
4,4,139495.201681


In [45]:
#downloading submission csv
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>